In [5]:
import sys
sys.path.insert(0, "/home/manas/Documents/skills/Beginner/math-reasoning-distill")

from src.data.normalize import build_clean_dataset
from src.eval.extract_answer import parse_numeric
from collections import Counter
import plotly.express as px
import plotly.graph_objects as go


In [15]:
gsm = build_clean_dataset("gsm8k", "train", n_samples=-1)
mm = build_clean_dataset("metamath", n_samples=8000)

[src/data/load_gsm8k.py] GSM8K train: 7473 examples
[src/data/normalize.py] build_clean_dataset: Clean gsm8k: 7473 examples
[src/data/load_metamath.py] MetaMathQA: 8000 examples


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

[src/data/normalize.py] build_clean_dataset: Clean metamath: 8000 examples


In [16]:
step_counts = Counter(gsm[i]["num_steps"] for i in range(len(gsm)))
fig = px.bar(
    x=sorted(step_counts.keys()),
    y=[step_counts[k] for k in sorted(step_counts.keys())],
    labels={"x": "Number of Steps", "y": "Count"},
    title="GSM8K — Problem Difficulty (Step Count)",
)
fig.show()

In [17]:
numeric = sum(1 for i in range(len(mm)) if parse_numeric(mm[i]["final_answer"]) is not None)
symbolic = len(mm) - numeric

fig = px.pie(
    names=["Numeric", "Symbolic"],
    values=[numeric, symbolic],
    title="MetaMath — Answer Types",
)
fig.show()


In [18]:
gsm_steps = Counter(gsm[i]["num_steps"] for i in range(len(gsm)))
mm_steps = Counter(mm[i]["num_steps"] for i in range(len(mm)))
all_steps = sorted(set(gsm_steps.keys()) | set(mm_steps.keys()))

fig = go.Figure()
fig.add_trace(go.Bar(name="GSM8K", x=all_steps, y=[gsm_steps.get(s, 0) for s in all_steps]))
fig.add_trace(go.Bar(name="MetaMath", x=all_steps, y=[mm_steps.get(s, 0) for s in all_steps]))
fig.update_layout(
    title="Step Distribution — GSM8K vs MetaMath",
    xaxis_title="Number of Steps",
    yaxis_title="Count",
    barmode="group",
)
fig.show()

In [19]:
for i in range(3):
    ex = gsm[i]
    print(f"\n--- {ex['id']} ({ex['num_steps']} steps) → Answer: {ex['final_answer']} ---")
    print(f"Q: {ex['question'][:120]}")
    for j, step in enumerate(ex["solution_steps"]):
        print(f"  Step {j+1}: {step[:80]}")



--- gsm8k_00000 (2 steps) → Answer: 72 ---
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natali
  Step 1: Natalia sold 48/2 = 24 clips in May.
  Step 2: Natalia sold 48+24 = 72 clips altogether in April and May.

--- gsm8k_00001 (2 steps) → Answer: 10 ---
Q: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
  Step 1: Weng earns 12/60 = $0.2 per minute.
  Step 2: Working 50 minutes, she earned 0.2 x 50 = $10.

--- gsm8k_00002 (3 steps) → Answer: 5 ---
Q: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided
  Step 1: In the beginning, Betty has only 100 / 2 = $50.
  Step 2: Betty's grandparents gave her 15 * 2 = $30.
  Step 3: This means, Betty needs 100 - 50 - 30 - 15 = $5 more.
